In [1]:
import os, json, sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')
print('API key set:', bool(os.getenv('ANTHROPIC_API_KEY')))

API key set: True


# Prompt Engineering Lab

**學習目標：**
- [ ] Zero-shot vs. Few-shot vs. CoT 對科學文本抽取的影響
- [ ] Self-consistency 如何提高數值抽取的可靠性
- [ ] Structured output (JSON mode) 的使用

**預計時間：** 3-4 小時

> 使用 Anthropic API + `claude-haiku-4-5`（便宜、支援 temperature、200K context）。`.env` 中要設定 `ANTHROPIC_API_KEY`。

## 1. 測試文本

用同一段文本比較不同 prompting 策略的效果

In [2]:
SAMPLE_TEXT = """
The perovskite oxide La₀.₈Sr₀.₂MnO₃ (LSMO) thin films were deposited via pulsed laser
deposition (PLD) at a substrate temperature of 700°C under an oxygen partial pressure
of 200 mTorr. Film thickness: approximately 50 nm. Post-deposition annealing at 800°C
for 2 hours. Metal-insulator transition at T_MI = 370 K.
"""
print(SAMPLE_TEXT)


The perovskite oxide La₀.₈Sr₀.₂MnO₃ (LSMO) thin films were deposited via pulsed laser
deposition (PLD) at a substrate temperature of 700°C under an oxygen partial pressure
of 200 mTorr. Film thickness: approximately 50 nm. Post-deposition annealing at 800°C
for 2 hours. Metal-insulator transition at T_MI = 370 K.



## 2. Zero-shot vs. CoT 比較

In [3]:
from chain_of_thought import zero_shot_extraction, zero_shot_cot_extraction, few_shot_cot_extraction
import anthropic
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env

# Zero-shot
result_zero = zero_shot_extraction(SAMPLE_TEXT, client)
print('Zero-shot:')
print(json.dumps(result_zero, indent=2, ensure_ascii=False))

Zero-shot:
{
  "material": "La₀.₈Sr₀.₂MnO₃",
  "deposition_method": "pulsed laser deposition",
  "substrate_temperature_C": 700,
  "oxygen_pressure_mTorr": 200,
  "film_thickness_nm": 50,
  "annealing_temperature_C": 800,
  "annealing_duration_h": 2,
  "MIT_temperature_K": 370,
  "room_temp_resistivity_ohm_cm": null
}


In [4]:
# Zero-shot CoT
result_cot = zero_shot_cot_extraction(SAMPLE_TEXT, client)
print('Zero-shot CoT:')
print(json.dumps(result_cot, indent=2, ensure_ascii=False))

Zero-shot CoT:
{
  "material": "La₀.₈Sr₀.₂MnO₃",
  "deposition_method": "pulsed laser deposition",
  "substrate_temperature_C": 700,
  "oxygen_pressure_mTorr": 200,
  "film_thickness_nm": 50,
  "annealing_temperature_C": 800,
  "annealing_duration_h": 2,
  "MIT_temperature_K": 370,
  "room_temp_resistivity_ohm_cm": null,
  "reasoning_notes": "All explicitly stated numerical parameters were extracted with their corresponding units. Substrate temperature and annealing temperature were given in °C and converted to numeric values. Oxygen pressure was given in mTorr as specified. Film thickness was approximately 50 nm. Metal-insulator transition temperature was clearly identified as 370 K. Room temperature resistivity was not mentioned in the text, so it is set to null."
}


In [5]:
# Few-shot CoT
result_fs = few_shot_cot_extraction(SAMPLE_TEXT, client)
print('Few-shot CoT:')
print(json.dumps(result_fs, indent=2, ensure_ascii=False))

Few-shot CoT:
{
  "material": "La₀.₈Sr₀.₂MnO₃ (LSMO)",
  "deposition_method": "pulsed laser deposition (PLD)",
  "substrate_temperature_C": 700,
  "oxygen_pressure_mTorr": 200,
  "film_thickness_nm": 50,
  "annealing_temperature_C": 800,
  "annealing_duration_h": 2,
  "MIT_temperature_K": 370,
  "room_temp_resistivity_ohm_cm": null,
  "reasoning_notes": "Substrate temperature 700°C is explicitly stated during PLD deposition. Oxygen partial pressure 200 mTorr is given as deposition condition. Film thickness 50 nm is explicitly stated. Post-deposition annealing at 800°C for 2 hours is clearly specified. Metal-insulator transition temperature T_MI = 370 K is provided. Room temperature resistivity is not mentioned in the text, so it is null."
}


## 3. Self-consistency (N=5 samples)

多路徑推理 + 多數決，觀察 confidence 分數。低 confidence 的欄位 = 未來 fine-tuning 的 hard cases。

In [6]:
from self_consistency import self_consistent_extraction
result_sc = self_consistent_extraction(SAMPLE_TEXT, client, n_samples=5)

Running 5 extraction samples with claude-sonnet-4-6...
  Sample 1/5: 10 fields extracted
  Sample 2/5: 10 fields extracted
  Sample 3/5: 10 fields extracted
  Sample 4/5: 10 fields extracted
  Sample 5/5: 10 fields extracted

--- Self-Consistency Results ---
✅ centrifuge_speed_rpm: None (confidence: 100%)
✅ nanorod_length_um: None (confidence: 100%)
✅ material: La₀.₈Sr₀.₂MnO₃ (LSMO) (confidence: 100%)
✅ drying_temperature_C: None (confidence: 60%)
✅ concentration_mM: None (confidence: 100%)
✅ synthesis_method: Pulsed Laser Deposition (PLD) (confidence: 100%)
✅ crystal_structure: perovskite (confidence: 100%)
✅ reaction_duration_h: 2 (confidence: 100%)
✅ reaction_temperature_C: 700 (confidence: 100%)
✅ nanorod_diameter_nm: None (confidence: 100%)


## 4. ReAct pattern（Reasoning + Acting）

看 agent 如何交替 `Thought → Action → Observation`。這是 Week 3 LangGraph agent 的雛形。

**新版 API（refactor 後）**：
- `react_agent(...)` 回傳 `Trajectory` 物件，不再只是 string
- 每次 run 自動 append 到 `trajectories.jsonl`
- 接受 `temperature` 參數，可做 sweep 實驗
- 詳見 `notes_trajectory_refactor.md`

In [ ]:
from react_pattern import react_agent
from trajectory import replay

question = """From this text, extract all experimental parameters and convert
oxygen pressure to Pa and substrate temperature to K:

'LSMO thin films were deposited via PLD at 700°C substrate temperature under
200 mTorr O₂. Film thickness was ~50 nm.'

Return as JSON."""

# react_agent now returns a Trajectory object and writes JSONL to ./trajectories.jsonl
traj = react_agent(question, client, temperature=0.0)

# Pretty-print the full reasoning trace (zero token cost)
replay(traj)

## 練習題

1. 修改 SAMPLE_TEXT 為一段你自己找到的材料科學論文摘要，比較三種方法的效果
2. 新增一個含有科學記號（如 5×10⁻⁷）的文本，觀察哪種方法處理得更好
3. 把 `MODEL = "claude-haiku-4-5"` 改成 `"claude-sonnet-4-6"`，看更強的模型差多少
4. 設計 domain-specific 的 few-shot 範例（LSMO、鈣鈦礦、ZnO...），看 few-shot 範例的領域相關性如何影響結果


In [8]:
# 你的實驗
MY_TEXT = """
1. Materials and ReagentsMethylammonium iodide (MAI) and Lead(II) iodide ($PbI_2$) were purchased from Sigma-Aldrich (99.9% purity). Anhydrous dimethyl sulfoxide (DMSO) and N,N-dimethylformamide (DMF) were used as solvents without further purification.2. Device FabricationThe perovskite films were deposited via a one-step solution process in a nitrogen-filled glovebox.Precursor Preparation: A 1.2 M precursor solution was prepared by dissolving $PbI_2$ and MAI (1:1 molar ratio) in a mixed solvent of DMF:DMSO (4:1 v/v).Spin-Coating: The solution was spun onto cleaned FTO/TiO₂ substrates at 4000 rpm for 30 seconds. During the last 10 seconds of spinning, 200 μL of chlorobenzene was dropped onto the center of the substrate as an antisolvent to induce rapid crystallization.Annealing: The resulting films were annealed at 100°C for 10 minutes on a hotplate to form a dense, polycrystalline morphology.3. CharacterizationStructural Analysis: The crystal structure and phase purity were characterized using X-ray Diffraction (XRD) with $Cu\ K\alpha$ radiation ($\lambda = 1.5406 \text{ \AA}$).Morphological Observation: Surface topography and cross-sectional images were obtained using a Field-Emission Scanning Electron Microscope (FE-SEM) operated at an accelerating voltage of 5 kV.Optical Measurements: UV-vis absorption spectra were recorded using a Shimadzu UV-2600 spectrophotometer. Steady-state photoluminescence (PL) spectra were acquired with an excitation wavelength of 450 nm.
"""

result = zero_shot_cot_extraction(MY_TEXT, client)
print(json.dumps(result, indent=2, ensure_ascii=False))

{
  "material": "MAPbI3 (Methylammonium Lead Iodide)",
  "deposition_method": "One-step solution spin-coating",
  "substrate_temperature_C": null,
  "oxygen_pressure_mTorr": null,
  "film_thickness_nm": null,
  "annealing_temperature_C": 100,
  "annealing_duration_h": 0.167,
  "MIT_temperature_K": null,
  "room_temp_resistivity_ohm_cm": null,
  "reasoning_notes": "Extracted parameters directly from text: annealing temperature is 100°C; annealing duration of 10 minutes converted to hours (10/60 = 0.167 h). Material identified as MAPbI3 from precursor composition (MAI:PbI2 1:1 molar ratio). Deposition method is one-step solution spin-coating at 4000 rpm for 30 seconds with chlorobenzene antisolvent. Parameters not mentioned in text (substrate temperature, oxygen pressure, film thickness, MIT temperature, resistivity) are marked as null. Additional experimental details noted: precursor concentration 1.2 M, solvent ratio DMF:DMSO 4:1 v/v, spin speed 4000 rpm, spin duration 30 s, antisolven

In [9]:
# Few-shot CoT
result_fs = few_shot_cot_extraction(MY_TEXT, client)
print('Few-shot CoT:')
print(json.dumps(result_fs, indent=2, ensure_ascii=False))

Few-shot CoT:
{
  "material": "MAPbI₃ (methylammonium lead iodide perovskite)",
  "deposition_method": "spin-coating (one-step solution process)",
  "substrate_temperature_C": null,
  "oxygen_pressure_mTorr": null,
  "film_thickness_nm": null,
  "annealing_temperature_C": 100,
  "annealing_duration_h": 0.167,
  "MIT_temperature_K": null,
  "room_temp_resistivity_ohm_cm": null,
  "reasoning_notes": "Material is MAPbI₃ perovskite from MAI and PbI₂ precursors. Deposition method is spin-coating at 4000 rpm for 30s with chlorobenzene antisolvent. Annealing temperature is 100°C for 10 minutes (converted to 0.167 hours). Substrate temperature during spin-coating not specified. No oxygen pressure, film thickness, MIT temperature, or resistivity data provided in the text. Characterization methods (XRD, FE-SEM, UV-vis, PL) are mentioned but no resulting measurements are reported."
}


### 在自己的文本上跑（含 temperature 比較）

對同一個 perovskite synthesis 段落，比較 `temperature=0.0` vs `temperature=0.4`，看 trajectory 步數 / token 差異。

In [ ]:
question = f"""From this text, extract all experimental parameters: {MY_TEXT}
"""
traj_t0 = react_agent(question, client, temperature=0.0)
print(f"\n→ {len(traj_t0.steps)} steps, {traj_t0.total_tokens} tokens, status={traj_t0.status}")

In [ ]:
traj_t04 = react_agent(question, client, temperature=0.4)
print(f"\n→ {len(traj_t04.steps)} steps, {traj_t04.total_tokens} tokens, status={traj_t04.status}")
print(f"\nDelta vs t=0: steps {len(traj_t04.steps) - len(traj_t0.steps):+d}, "
      f"tokens {traj_t04.total_tokens - traj_t0.total_tokens:+d}")

## 5. Temperature Sweep 實驗

跑 `4 個 temperature × 3 trial = 12 條 trajectory`，看 temperature 怎麼影響：
- `success_rate`：工具呼叫格式被破壞的比例
- `avg_steps` / `std_steps`：步數平均 + 變異度（**std 才是真正的 sensor**）
- `avg_tokens`：對應成本

> ⏱ 12 條 trajectory，每條 3-6 步，預估 ~2 分鐘 + ~$0.05 (Haiku)
> 跑完之後 `trajectories.jsonl` 會多 12 行，可以重複 `analyze_trajectories` 看歷史。

In [ ]:
from react_pattern import run_temperature_sweep

# 跑 4 個 temp x 3 trials = 12 條 trajectory，全寫進 trajectories.jsonl
trajs = run_temperature_sweep(
    temperatures=[0.0, 0.3, 0.7, 1.0],
    trials=3,
)
print(f"\nTotal: {len(trajs)} trajectories")

In [ ]:
# 用 analyze_trajectories.py 的內部函數做 inline 分析（純 stdlib，不用 pandas）
from analyze_trajectories import load, step_metrics, summarize, _print_table

# 只看本次 sweep 的 runs（過濾 metadata 有 sweep_trial 的）
all_trajs = load('trajectories.jsonl')
sweep_trajs = [t for t in all_trajs if 'sweep_trial' in (t.get('metadata') or {})]
rows = [step_metrics(t) for t in sweep_trajs]

print(f'=== {len(rows)} sweep runs ===\n')
_print_table(summarize(rows))

In [ ]:
# 看每個 temperature 下的 status 分佈（哪些 t 開始壞掉）
from collections import Counter

by_temp_status = Counter((r['temperature'], r['status']) for r in rows)
print(f'{"temp":<6} {"status":<14} {"count":<5}')
for (t, s), n in sorted(by_temp_status.items()):
    print(f'{t:<6} {s:<14} {n:<5}')

In [ ]:
# 挑一條失敗 / 步數最多的 run，replay 它看模型在想什麼
from trajectory import Trajectory, Step, replay

# 找出步數最多的那條
worst = max(sweep_trajs, key=lambda t: len(t['steps']))

# 把 dict 還原成 Trajectory 物件
traj_obj = Trajectory(
    run_id=worst['run_id'],
    question=worst['question'],
    final_answer=worst.get('final_answer'),
    status=worst['status'],
    steps=[Step(**s) for s in worst['steps']],
    total_tokens=worst['total_tokens'],
    total_latency_ms=worst['total_latency_ms'],
    model=worst['model'],
    temperature=worst['temperature'],
    metadata=worst['metadata'],
    created_at=worst['created_at'],
)
replay(traj_obj)

## 觀察與下一步

你應該會看到（典型結果）：
- `t=0.0` 步數低 + std=0 + 100% success
- `t=0.7` 開始有 `tool_error` / `parse_error`，步數變異大
- `t=1.0` success_rate 明顯下降

**下一步建議**：
1. 換一個更難的 question（要多步檢索 + 抽取 + 計算），sweep 結果會更明顯
2. 把 `MODEL` 換成 `claude-sonnet-4-6` 重跑同樣 sweep，看模型強度對 temperature robustness 的影響
3. （Week 5 預習）寫一個 `trajectory_to_sft_messages(traj)` 函數，把成功的 trajectory 轉成 SFT 訓練格式